# 55장 · create_agent의 도구 등록과 메시지 왕복

create_agent는 모델 호출과 도구 실행을 연결합니다. 대화 메시지에서 도구 요청과 결과를 찾아 어느 함수가 실행됐는지 확인합니다.

17장에서 직접 작성한 도구 호출 루프를 create_agent로 구성합니다. 호출을 지정한 테스트 모델로 주말 운영 시간과 대출 기간을 요청하고, 도구 메시지와 반환값을 대조합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain 1.4.6, langchain-core 1.6.12, langsmith 0.14.11입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 도구 요청과 결과 메시지 연결

도구 요청에는 함수 이름과 인자, 결과를 받을 때 사용할 호출 ID가 들어 있습니다. 시간을 알아내는 일은 등록된 함수가 맡습니다. 이 예제에서는 미리 지정한 도구 요청을 Python 함수에 전달합니다.


### 셀 01 · 도구 함수의 인자와 결과

요청에 시간표 값이 있는지 먼저 보세요. 조회 뒤 어느 값이 새로 생길까요?


In [1]:
hours = {"weekday": "09:00~18:00", "weekend": "10:00~16:00"}
request = {"name": "get_hours", "args": {"day": "weekend"}, "id": "c1"}
def get_hours(day):
    return {"ok": True, "hours": hours[day]}

result = get_hours(**request["args"])
print(result["hours"])

10:00~16:00


10:00~16:00은 함수가 시간표에서 꺼낸 값입니다. 요청의 c1은 시간이 들어갈 칸이 아니라 도구 결과를 원래 요청과 짝지을 식별자입니다.


### 과제 02 · 요청과 결과의 ID 연결

두 번째 요청 c2의 응답에서 대출 기간을 꺼내세요. 목록 위치 대신 호출 ID를 쓰세요.


In [2]:
replies = {'c1': {'hours': '10:00~16:00'}, 'c2': {'loan_days': 14}}
# c2의 대출 기간을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>replies = {&#x27;c1&#x27;: {&#x27;hours&#x27;: &#x27;10:00~16:00&#x27;}, &#x27;c2&#x27;: {&#x27;loan_days&#x27;: 14}}
print(replies[&#x27;c2&#x27;][&#x27;loan_days&#x27;])</code></pre>
<p>실행 결과</p><pre><code>14</code></pre>
<p>14는 대출 규정 도구가 반환한 일수입니다. 반환 순서가 달라져도 c2로 찾아낼 수 있습니다.</p>
</details>


## 대화 기록에서 도구 호출 수 집계

먼저 제공된 메시지 유형 목록에서 ai와 tool을 따로 셉니다. 뒤의 셀 06에서는 두 함수를 create_agent에 등록해 실제 ToolMessage의 이름·ID·반환값을 읽습니다.


### 셀 03 · 도구 메시지 수 집계

대본에서 나온 메시지 종류입니다. tool이 등장한 수와 ai가 등장한 수를 따로 확인하세요.


In [3]:
roles = ["human", "ai", "tool", "ai", "tool", "ai"]
print(roles.count("tool"), roles.count("ai"))

2 3


첫 값 2는 도구 결과 메시지 수이고 둘째 값 3은 AI 메시지 수입니다. 이 목록에서는 각 AI 요청이 도구 하나만 부릅니다. 한 AI 메시지가 여러 도구를 요청할 수도 있으므로 메시지 수를 항상 호출 수로 간주할 수는 없습니다.


### 역할별 메시지 세기

<p>roles에서 ai는 세 번, tool은 두 번 나타납니다. 마지막 ai는 두 도구의 결과를 받은 뒤 답변만 쓰기 때문에 ToolMessage가 하나 더 생기지 않습니다.</p>


### 과제 04 · 대응 결과가 없는 호출

c1과 c2를 요청했지만 c1만 회신했습니다. 회신을 기다리는 ID를 출력하세요.


In [4]:
requested = ['c1', 'c2']
returned = {'c1'}
# 아직 결과가 없는 ID를 찾으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>requested = [&#x27;c1&#x27;, &#x27;c2&#x27;]
returned = {&#x27;c1&#x27;}
print([call_id for call_id in requested if call_id not in returned])</code></pre>
<p>실행 결과</p><pre><code>[&#x27;c2&#x27;]</code></pre>
<p>[&#x27;c2&#x27;]가 남습니다. 이 상태에서 대출 기간을 확정하면 도구가 주지 않은 답을 쓰게 됩니다.</p>
</details>


## create_agent에 두 도구 등록

이 절은 LangChain 1.x 수업 커널에서 실행합니다. 대본 모델은 질문의 의미를 판단하는 대신 정해 둔 순서대로 요청을 냅니다. 도구를 찾아 실행하고 그 결과를 메시지에 붙이는 부분은 실제 LangChain이 맡습니다.

### 셀 06에서 두 도구의 반환값은 어디로 갈까요?

**주말 시간표를 요청합니다.**　첫 모델 메시지에는 도구 이름과 요청 ID가 있습니다.

```
c1 · get_hours
```

**함수가 시간표를 돌려줍니다.**　도구 메시지의 tool_call_id는 첫 요청의 ID인 c1입니다.

```
c1 · 10:00~16:00
```

**대출 기간을 요청합니다.**　다음 모델 메시지가 두 번째 요청을 냅니다.

```
c2 · get_loan_rule
```

**함수가 대출 기간을 돌려줍니다.**　이 값은 c2가 요청한 대출 일수입니다.

```
c2 · 14
```

**마지막 모델 메시지가 답을 냅니다.**　추가 도구 요청이 없어 왕복이 끝납니다.

```
모델 메시지 3개 · 도구 메시지 2개
```


### 셀 05 · 도구 요청을 내는 테스트 모델

도구 요청을 담은 메시지를 만들 도우미입니다. 다음 셀에서 쓰므로 출력은 없습니다.


In [5]:
from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langsmith import tracing_context

class ClassroomModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, **kwargs):
        return self  # 사용할 도구 요청은 아래 대본에 고정했습니다.

def ask_tool(name, args, call_id):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id}])

bind_tools가 대본 모델을 그대로 돌려줍니다. 네트워크나 결제 계정 없이 같은 요청 순서를 반복해서 볼 수 있습니다.


### 모델과 도구 사이에서 누가 함수를 실행할까?

<p>33장에서 직접 연결한 도구 실행과 다음 모델 호출을 create_agent가 이어 줍니다. 이번에는 마지막 AIMessage의 tool_calls가 비는 시점까지 따라가세요.</p>


### 같은 도구 요청을 다시 관찰하기

<p>ClassroomModel이 내놓을 세 응답은 코드에 미리 적혀 있습니다. 그래서 노트북을 다시 실행해도 첫 도구는 get_hours, 두 번째 도구는 get_loan_rule입니다. 여기서 비교할 수 있는 것은 실행 순서입니다.</p>


### 같은 모델로 돌아가는 두 도구 요청

두 도구는 차례로 실행됩니다. 결과를 모델에게 돌려줄 때까지 c1과 c2가 각각 어느 요청의 ID인지 따라가세요.

1. 고정 모델 · 첫 요청 ID = c1
2. get_hours · c1 반환 = 10:00~16:00
3. 고정 모델 · 다음 요청 ID = c2
4. get_loan_rule · c2 반환 = 14
5. 고정 모델 · 다음 도구 요청 수 = 0
6. 최종 응답 · 답변의 두 값 = 10:00~16:00 · 14일


### 셀 06 · create_agent의 도구 왕복

각 도구가 실제로 불렸을 때의 반환값을 호출 ID와 함께 읽습니다.

셀 01의 같은 이름 함수는 사전을 반환해서 hours 키를 읽었습니다. 여기에서 새로 정의하는 도구는 문자열을 돌려줍니다. 메시지에서 호출 ID를 먼저 찾은 뒤 그 ID와 짝을 이룬 ToolMessage의 content를 읽으세요.


In [6]:
def get_hours(day: str) -> str:
    """가상 도서관의 운영 시간을 찾는다."""
    return hours[day]

def get_loan_rule() -> str:
    """가상 도서관 대출 일수를 찾는다."""
    return "14"

model = ClassroomModel(responses=[
    ask_tool("get_hours", {"day": "weekend"}, "c1"),
    ask_tool("get_loan_rule", {}, "c2"),
    AIMessage(content="주말은 10:00~16:00이며 대출은 14일입니다."),
])
agent = create_agent(model, tools=[get_hours, get_loan_rule])
with tracing_context(enabled=False):
    state = agent.invoke({"messages": [("user", "주말 시간과 대출 기간은?")]})
for message in state["messages"]:
    if message.type == "tool":
        print(message.tool_call_id, message.content)

c1 10:00~16:00
c2 14


첫 도구 메시지에는 c1과 주말 시간, 두 번째 메시지에는 c2와 대출 기간 14가 들어 있습니다.

이 예제의 테스트 모델에는 도구 호출이 미리 지정돼 있습니다. 반환된 메시지에서 검증할 수 있는 것은 요청이 실제 함수 실행으로 이어졌는지, 그 결과가 같은 호출 ID로 돌아왔는지입니다. 새 질문에 어떤 도구를 고를지는 이 기록만으로 평가할 수 없습니다.


#### 도구 두 개를 등록하면 둘 다 실행될까요?

<mark>등록은 호출할 수 있는 함수 목록을 정하고 실제 순서는 모델 메시지의 tool_calls가 정합니다.</mark> 이 실습의 테스트 모델은 c1으로 주말 시간, c2로 대출 일수를 차례로 요청합니다. 마지막 AI 메시지에는 새 요청이 없어 반복이 끝납니다. 결과 메시지 두 개와 마지막 답변에 쓰인 값을 짝지어 보세요.

선수 개념 · 33장 LangChain 검색 도구와 ToolMessage

#### create_agent의 두 번의 도구 왕복

```mermaid
flowchart TD
s["사용자 질문"] --> m["모델 호출"]
m --> d{"이번 AIMessage의 tool_calls?"}
d -->|c1| h["get_hours · 주말 시간"]
d -->|c2| l["get_loan_rule · 대출 일수"]
h --> t["ToolMessage · 호출 ID와 결과"]
l --> t
t --> m
d -->|없음| e["최종 AIMessage"]
```

이 테스트 모델이 보낸 순서는 c1, c2, 최종 답변입니다. 함수가 반환한 값은 ToolMessage로 대화에 들어가고 모델이 그 대화를 다시 읽습니다.

## 복습과 실습 점검

### 과제 07 · 도구 결과에서 최종 답변에 쓰인 값 찾기

셀 06이 반환한 state에서 호출 ID가 c2인 도구 메시지를 찾아 도구 이름과 반환값을 출력하세요. 메시지의 위치는 실행 경로에 따라 달라질 수 있습니다.


In [ ]:
# state["messages"]에서 tool_call_id가 "c2"인 도구 결과를 찾으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>for message in state[&quot;messages&quot;]:
    if message.type == &quot;tool&quot; and message.tool_call_id == &quot;c2&quot;:
        print(message.name, message.content)</code></pre>
<p>실행 결과</p><pre><code>get_loan_rule 14</code></pre>
<p>c2가 가리키는 도구는 대출 규정을 조회했습니다. 마지막 AI 메시지의 문장을 바꾸더라도 이 ToolMessage에는 실행한 함수의 반환값이 남습니다.</p>
</details>


## create_agent가 검색 도구를 다시 부르는 순간

33장에서는 조건 간선과 도구 노드를 직접 연결했습니다. 여기서는 create_agent로 만든 실제 그래프에 한 번 질문한 뒤, 모델과 도구 중 어느 쪽이 다음 메시지를 붙였는지 노드 업데이트에서 찾습니다. ClassroomModel은 질문마다 선택 순서를 재현하는 수업용 대역입니다.

### 셀 09 · 도구 호출 전후의 노드 업데이트 읽기

주말 시간과 대출 기간을 물었던 agent를 같은 입력으로 실행합니다. 최종 상태 한 덩어리를 읽었던 invoke와 달리 stream의 updates에서는 방금 일한 노드 이름이 보입니다.

In [7]:
question_input = {"messages": [("user", "주말 시간과 대출 기간은?")]}
with tracing_context(enabled=False):
    for update in agent.stream(question_input, stream_mode="updates"):
        for node_name, change in update.items():
            message = change["messages"][-1]
            if message.type == "tool":
                print(node_name, message.tool_call_id, message.content)
            elif message.tool_calls:
                print(node_name, message.tool_calls[0]["name"])
            else:
                print(node_name, message.content)

model get_hours
tools c1 10:00~16:00
model get_loan_rule
tools c2 14
model 주말은 10:00~16:00이며 대출은 14일입니다.


처음에는 model이 get_hours를 요청하고 tools가 c1의 운영 시간을 돌려줍니다. 이어 model이 get_loan_rule을 요청하고 tools가 c2에 14를 붙입니다. 마지막 model의 메시지에는 새 호출 요청이 없어 실행이 끝납니다. 도구 이름은 실행 전 요청에도 보이므로 실제 실행 여부는 tools의 결과로 확인하세요.

### 과제 10 · 같은 도구를 등록하고도 검색 없이 끝내기

인사 한 건에는 도구 요청이 없는 AIMessage 한 건만 준비하세요. 기존 도구 두 개를 똑같이 등록한 새 에이전트를 만들고, 인사를 보낸 뒤 전체 메시지 타입과 마지막 문장을 출력하세요.

In [8]:
greeting_input = {"messages": [("user", "안녕하세요!")]}
# 응답 한 건만 준비한 ClassroomModel과 기존 도구로 새 에이전트를 만드세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>greeting_input = {&quot;messages&quot;: [(&quot;user&quot;, &quot;안녕하세요!&quot;)]}
greeting_model = ClassroomModel(responses=[AIMessage(content=&quot;도서관 안내입니다.&quot;)])
greeting_agent = create_agent(greeting_model, tools=[get_hours, get_loan_rule])
with tracing_context(enabled=False):
    history = greeting_agent.invoke(greeting_input)[&quot;messages&quot;]
print([message.type for message in history])
print(history[-1].content)</code></pre>
<p>실행 결과</p><pre><code>[&#x27;human&#x27;, &#x27;ai&#x27;]
도서관 안내입니다.</code></pre>
<p>human, ai 두 메시지가 남고 도서관 안내입니다가 출력됩니다. 같은 도구 두 개를 등록해도 모델 응답이 호출을 요청하지 않으면 tools 노드를 지나지 않습니다. 이 대역은 대본대로 답하므로 인사 의도를 얼마나 잘 분류하는지 평가하는 데는 쓸 수 없습니다.</p>
</details>

### 셀 11 · 실제 모델의 선택을 확인하는 선택 셀

도구 호출을 지원하는 수업용 채팅 모델을 준비했다면 class_chat에 넣으세요. OpenAI 계정으로 실습할 때는 커널에 langchain-openai와 OPENAI_API_KEY를 설정하고 주석의 모델 이름을 수업에서 쓰는 값으로 바꾸세요. 기본 설정에서는 외부 요청을 보내지 않습니다.

In [9]:
class_chat = None
# from langchain_openai import ChatOpenAI
# class_chat = ChatOpenAI(model="수업에서 지정한 모델명")
if class_chat is None:
    print("채팅 모델을 연결하면 도구 선택을 확인할 수 있습니다.")
else:
    live_agent = create_agent(class_chat, tools=[get_hours, get_loan_rule])
    with tracing_context(enabled=False):
        run = live_agent.invoke(question_input)
    for message in run["messages"]:
        if message.type == "tool":
            print(message.name, message.content)
    print(run["messages"][-1].content)

채팅 모델을 연결하면 도구 선택을 확인할 수 있습니다.


실제 모델을 연결했다면 최종 문장에 운영 시간이 있는지와 get_hours의 결과가 기록에 있는지를 각각 확인하세요. 답에 시간이 있어도 도구를 거쳤다는 증거는 그 앞의 ToolMessage에서 찾아야 합니다.

<mark>도구를 등록한 횟수와 도구를 실행한 횟수는 다른 값입니다.</mark> 대본을 바꿔도 등록 목록은 같습니다. 두 실습에서 달라진 것은 모델이 반환한 AIMessage의 tool_calls와 그 결과로 지나간 노드입니다.

56장에서는 모델이 요청한 호출 중 정책이 실제 함수 실행을 허용한 건수를 따로 확인합니다.